In [2]:
import pandas as pd
df=pd.read_csv("../data/processed/abd_vs_bowlers.csv")


In [3]:
# creating/defining our phases

def classify_phase(over):
    if 0<=over<=5:
        return 'Powerplay'
    elif 6<=over<=14:
        return 'Middle Overs'
    elif  15<=over <=19:
        return 'Death Overs'
    else:
        return "Unknown"

df['phase']=df['overs'].apply(classify_phase)

In [4]:
df[['overs','phase']].head(10)

,overs,phase
0,12,Middle Overs
1,12,Middle Overs
2,12,Middle Overs
3,14,Middle Overs
4,14,Middle Overs
5,13,Middle Overs
6,13,Middle Overs
7,16,Death Overs
8,16,Death Overs
9,16,Death Overs


In [5]:
df['phase'].value_counts()

phase
Middle Overs    233
Death Overs     128
Powerplay        57
Name: count, dtype: int64

In [6]:
# creating dot balls

df['dot_ball']=(df['total_run']==0).astype(int)


In [7]:
df[['total_run','dot_ball']].head(15)

,total_run,dot_ball
0,1,0
1,0,1
2,4,0
3,2,0
4,0,1
5,1,0
6,1,0
7,0,1
8,6,0
9,1,0


In [8]:
# creating dismissals involving target batter and not in run-out form

df['is_dismissal']=((df['player_out']=="AB de Villiers") &(df['kind']!='run out')).astype(int)

In [9]:
df[['ballnumber','is_dismissal']].head(15)

,ballnumber,is_dismissal
0,3,0
1,5,0
2,6,0
3,1,0
4,2,1
5,3,0
6,6,0
7,1,0
8,2,0
9,3,0


In [10]:
# validation
df[df['is_dismissal'] == 1][
    ['bowler', 'player_out', 'kind']
]


,bowler,player_out,kind
4,SP Narine,AB de Villiers,bowled
13,JJ Bumrah,AB de Villiers,caught
22,RA Jadeja,AB de Villiers,bowled
66,RA Jadeja,AB de Villiers,caught
103,SP Narine,AB de Villiers,caught
137,SP Narine,AB de Villiers,bowled
257,JJ Bumrah,AB de Villiers,caught
264,SP Narine,AB de Villiers,bowled
277,JJ Bumrah,AB de Villiers,bowled
353,R Ashwin,AB de Villiers,stumped


In [11]:
# checking how extras are stored 
df['extra_type'].unique()

<StringArray>
[nan, 'wides', 'legbyes', 'byes', 'noballs']
Length: 5, dtype: str

In [12]:
# creating legal deliveries
df['is_legal_ball']=(~df['extra_type'].isin(['wides'])).astype(int)


In [13]:
# validation
df[['extra_type','is_legal_ball']].head(10)

,extra_type,is_legal_ball
0,NaN,1
1,NaN,1
2,NaN,1
3,NaN,1
4,NaN,1
5,NaN,1
6,NaN,1
7,NaN,1
8,NaN,1
9,NaN,1


In [66]:
# creating fours hit 
df['4s']=(df['batsman_run']==4).astype(int)

In [67]:
df['6s']=(df['batsman_run']==6).astype(int)

In [74]:
# grouping by bowlers vs abd stats

stats=df.groupby('bowler').agg({
    'batsman_run':'sum',
    'is_legal_ball':'sum',
    'dot_ball':'sum',
    'is_dismissal':'sum',
    '4s':'sum',
    '6s':'sum'
    
}).reset_index()


In [75]:
stats.head(6)

,bowler,batsman_run,is_legal_ball,dot_ball,is_dismissal,4s,6s
0,DJ Bravo,72,45,8,1,8,2
1,DW Steyn,55,23,5,1,3,5
2,JJ Bumrah,131,89,33,3,10,8
3,R Ashwin,69,61,15,2,5,2
4,RA Jadeja,111,91,29,2,8,5
5,SL Malinga,124,61,14,0,10,8


In [76]:
# derving our metrics now
# starting with Strike Rate


stats['strike_rate']=((stats['batsman_run']/stats['is_legal_ball'])*100).round(2)

In [77]:
stats.head(6)

,bowler,batsman_run,is_legal_ball,dot_ball,is_dismissal,4s,6s,strike_rate
0,DJ Bravo,72,45,8,1,8,2,160.00
1,DW Steyn,55,23,5,1,3,5,239.13
2,JJ Bumrah,131,89,33,3,10,8,147.19
3,R Ashwin,69,61,15,2,5,2,113.11
4,RA Jadeja,111,91,29,2,8,5,121.98
5,SL Malinga,124,61,14,0,10,8,203.28


In [78]:
# deriving dot ball %

stats['dot_ball%']=((stats['dot_ball']/stats['is_legal_ball'])*100).round(2)

In [79]:
stats.head(6)

,bowler,batsman_run,is_legal_ball,dot_ball,is_dismissal,4s,6s,strike_rate,dot_ball%
0,DJ Bravo,72,45,8,1,8,2,160.00,17.78
1,DW Steyn,55,23,5,1,3,5,239.13,21.74
2,JJ Bumrah,131,89,33,3,10,8,147.19,37.08
3,R Ashwin,69,61,15,2,5,2,113.11,24.59
4,RA Jadeja,111,91,29,2,8,5,121.98,31.87
5,SL Malinga,124,61,14,0,10,8,203.28,22.95


In [80]:
# deriving batting average

stats['batting_avg']=(stats['batsman_run']/stats['is_dismissal']).round(2)

In [81]:
stats.head(6)

,bowler,batsman_run,is_legal_ball,dot_ball,is_dismissal,4s,6s,strike_rate,dot_ball%,batting_avg
0,DJ Bravo,72,45,8,1,8,2,160.00,17.78,72.00
1,DW Steyn,55,23,5,1,3,5,239.13,21.74,55.00
2,JJ Bumrah,131,89,33,3,10,8,147.19,37.08,43.67
3,R Ashwin,69,61,15,2,5,2,113.11,24.59,34.50
4,RA Jadeja,111,91,29,2,8,5,121.98,31.87,55.50
5,SL Malinga,124,61,14,0,10,8,203.28,22.95,inf


In [82]:
# creating phase-wise stats

phase_stats=df.groupby(['bowler','phase']).agg({
        'batsman_run':'sum',
        'is_legal_ball':'sum',
        'dot_ball':'sum',
        'is_dismissal':'sum',
        'is_boundary':'sum',
        '4s':'sum',
        '6s':'sum'
    }).reset_index()


In [83]:
phase_stats.head(21)

,bowler,phase,batsman_run,is_legal_ball,dot_ball,is_dismissal,is_boundary,4s,6s
0,DJ Bravo,Death Overs,22,10,1,0,3,1,2
1,DJ Bravo,Middle Overs,33,23,2,1,4,4,0
2,DJ Bravo,Powerplay,17,12,5,0,3,3,0
3,DW Steyn,Death Overs,47,15,2,0,8,3,5
4,DW Steyn,Middle Overs,5,7,3,1,0,0,0
5,DW Steyn,Powerplay,3,1,0,0,0,0,0
6,JJ Bumrah,Death Overs,80,36,8,1,13,7,6
7,JJ Bumrah,Middle Overs,36,31,12,2,3,1,2
8,JJ Bumrah,Powerplay,15,22,13,0,2,2,0
9,R Ashwin,Death Overs,21,14,2,0,3,2,1


In [84]:
# deriving metrics for phase_stats
# Strike Rate

phase_stats['strike_rate']=((phase_stats['batsman_run']/phase_stats['is_legal_ball'])*100).round(2)

In [85]:
# deriving phase wise dot ball %
phase_stats['dot_ball_pct']=(phase_stats['dot_ball']/phase_stats['is_legal_ball']*100).round(2)

In [86]:
phase_stats.head(21)

,bowler,phase,batsman_run,is_legal_ball,dot_ball,is_dismissal,is_boundary,4s,6s,strike_rate,dot_ball_pct
0,DJ Bravo,Death Overs,22,10,1,0,3,1,2,220.00,10.00
1,DJ Bravo,Middle Overs,33,23,2,1,4,4,0,143.48,8.70
2,DJ Bravo,Powerplay,17,12,5,0,3,3,0,141.67,41.67
3,DW Steyn,Death Overs,47,15,2,0,8,3,5,313.33,13.33
4,DW Steyn,Middle Overs,5,7,3,1,0,0,0,71.43,42.86
5,DW Steyn,Powerplay,3,1,0,0,0,0,0,300.00,0.00
6,JJ Bumrah,Death Overs,80,36,8,1,13,7,6,222.22,22.22
7,JJ Bumrah,Middle Overs,36,31,12,2,3,1,2,116.13,38.71
8,JJ Bumrah,Powerplay,15,22,13,0,2,2,0,68.18,59.09
9,R Ashwin,Death Overs,21,14,2,0,3,2,1,150.00,14.29


In [ ]:
# this won't contain our is_boundary which was created later
phase_stats.drop(columns=['batting_avg'])

,bowler,phase,batsman_run,is_legal_ball,dot_ball,is_dismissal,strike_rate,dot_ball_pct
0,DJ Bravo,Death Overs,22,10,1,0,220.00,10.00
1,DJ Bravo,Middle Overs,33,23,2,1,143.48,8.70
2,DJ Bravo,Powerplay,17,12,5,0,141.67,41.67
3,DW Steyn,Death Overs,47,15,2,0,313.33,13.33
4,DW Steyn,Middle Overs,5,7,3,1,71.43,42.86
5,DW Steyn,Powerplay,3,1,0,0,300.00,0.00
6,JJ Bumrah,Death Overs,80,36,8,1,222.22,22.22
7,JJ Bumrah,Middle Overs,36,31,12,2,116.13,38.71
8,JJ Bumrah,Powerplay,15,22,13,0,68.18,59.09
9,R Ashwin,Death Overs,21,14,2,0,150.00,14.29


#### we derived phase-wise batting avg but dropped as it's but irrelevant to bring in Avg phase-wise as it lacks depth

In [87]:
# instead deriving phase_wise boundary pct
# it is balls hit to boundary not % of runs scored in boundaries

df['is_boundary']=((df['batsman_run']==4) | (df['batsman_run']==6)).astype(int)

In [88]:
df['is_boundary'].head(20)

0     0
1     0
2     1
3     0
4     0
5     0
6     0
7     0
8     1
9     0
10    0
11    1
12    0
13    0
14    0
15    0
16    1
17    0
18    0
19    0
Name: is_boundary, dtype: int64

#### we just added this column in the cell where we first created phase_stats and re-run the cells to get what we needed 

In [89]:
phase_stats['boundary_pct']=((phase_stats['is_boundary']/phase_stats['is_legal_ball'])*100).round(2)

In [92]:
phase_stats.head()

,bowler,phase,batsman_run,is_legal_ball,dot_ball,is_dismissal,is_boundary,4s,6s,strike_rate,dot_ball_pct,boundary_pct
0,DJ Bravo,Death Overs,22,10,1,0,3,1,2,220.00,10.00,30.00
1,DJ Bravo,Middle Overs,33,23,2,1,4,4,0,143.48,8.70,17.39
2,DJ Bravo,Powerplay,17,12,5,0,3,3,0,141.67,41.67,25.00
3,DW Steyn,Death Overs,47,15,2,0,8,3,5,313.33,13.33,53.33
4,DW Steyn,Middle Overs,5,7,3,1,0,0,0,71.43,42.86,0.00


In [93]:
stats.to_csv("../data/processed/abd_vs_bowlers_overall.csv",index=False)

In [94]:
phase_stats.to_csv("../data/processed/abd_vs_bowler_phase-wise.csv",index=False)